In [ ]:
from IPython.core.display import display, HTML, Javascript

# ----- Notebook Theme -----
color_map = ['#6166B3', '#e8eff6', '#d0deec', '#a2bdd9', '#a2b9d9', '#4568b3',
                        '#6166B3', '#13488d', '#11397a', '#113e7a', '#0b2553']

prompt = color_map[-1]
main_color = color_map[0]
strong_main_color = color_map[1]
custom_colors = [strong_main_color, main_color]

css_file = '''

div #notebook {
background-color: white;
line-height: 20px;
}

#notebook-container {
%s
margin-top: 2em;
padding-top: 2em;
border-top: 4px solid %s; /* light orange */
-webkit-box-shadow: 0px 0px 8px 2px rgba(224, 212, 226, 0.5); /* pink */
    box-shadow: 0px 0px 8px 2px rgba(224, 212, 226, 0.5); /* pink */
}

div .input {
margin-bottom: 1em;
}

.rendered_html h1, .rendered_html h2, .rendered_html h3, .rendered_html h4, .rendered_html h5, .rendered_html h6 {
color: %s; /* light orange */
font-weight: 600;
}

div.input_area {
border: none;
    background-color: %s; /* rgba(229, 143, 101, 0.1); light orange [exactly #E58F65] */
    border-top: 2px solid %s; /* light orange */
}

div.input_prompt {
color: %s; /* light blue */
}

div.output_prompt {
color: %s; /* strong orange */
}

div.cell.selected:before, div.cell.selected.jupyter-soft-selected:before {
background: %s; /* light orange */
}

div.cell.selected, div.cell.selected.jupyter-soft-selected {
    border-color: %s; /* light orange */
}

.edit_mode div.cell.selected:before {
background: %s; /* light orange */
}

.edit_mode div.cell.selected {
border-color: %s; /* light orange */

}
'''
def to_rgb(h):
    return tuple(int(h[i:i+2], 16) for i in [0, 2, 4])

main_color_rgba = 'rgba(%s, %s, %s, 0.1)' % (to_rgb(main_color[1:]))
open('notebook.css', 'w').write(css_file % ('width: 95%;', main_color, main_color, main_color_rgba, main_color,  main_color, prompt, main_color, main_color, main_color, main_color))

def nb():
    return HTML("<style>" + open("notebook.css", "r").read() + "</style>")
nb()

<h1 style="font-size:80px;color:#6166B3;text-align:center"><strong>Chest X-Ray</strong> <strong style="color:black">Image Classification</strong></h1>

<p style="font-size:120%"><strong>Pneumonia</strong> is an infection that causes inflammation in one or both of the lungs and may be caused by a virus, bacteria, fungi or other germs.</p>

<p style="font-size:120%"><mark>Your doctor may conduct a physical exam and <strong>use chest x-ray,</strong> chest CT, chest ultrasound, or needle biopsy of the lung to help diagnose your condition.</mark> Your doctor may further evaluate your condition and lung function using thoracentesis, chest tube placement or image-guided abscess drainage.</p>

<p style="font-size:120%">Reference: <a href="https://www.radiologyinfo.org/en/info/pneumonia">https://www.radiologyinfo.org/en/info/pneumonia</a></p>

![image.png](https://img.grepmed.com/uploads/5678/pneumonia-cxr-comparison-lung-clinical-original.gif)

<p style="font-size:120%">This is where Machine Learning comes into picture! What if <mark><strong>we can build an ML model that can identify Chest X-Rays!</strong></mark> This can help reduce the time it takes for the Radiologists and Doctors to evaluate the X-Rays. This can save time and the patient can be treated soon.</p>

<p style="font-size:120%">Therefore, let's try to tackle this use-case in this notebook :) </p>

<h1 style="font-size:50px;color:#6166B3"><strong>Importing </strong><strong style="color:black">Necessary Libraries:</strong></h1>

In [ ]:
# Importing OS and Pathlib, so that we can work with files in our directories/PC
import os
import pathlib

# Some basic libraries that help us create dataframes and visualizations.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import image
plt.style.use("fivethirtyeight")
%matplotlib inline

# Deep Learning Libraries like Tensorflow and Keras
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from tensorflow.keras import layers
from keras.layers import Dense, Flatten

# Shows the Tensorflow and Keras Version
print("Tensorflow Version:", tf.__version__)
print("Keras Version:", keras.__version__)

# To Suppress any Un-necessary warnings
import warnings
warnings.filterwarnings('ignore')

<h1 style="font-size:50px;color:#6166B3"><strong>Getting </strong><strong style="color:black">Our Data:</strong></h1>

<p style="font-size:120%">Currently, we have the images in the following way:</p>

![image.png](attachment:0a6bf81d-690c-4110-8252-add7521d35e0.png)

In [ ]:
# Here, we define the paths where we have our data for training, testing and predictions
data_dir_train = pathlib.Path('../input/beginner-chest-xray-image-classification/chest_xray/train')
data_dir_test = pathlib.Path('../input/beginner-chest-xray-image-classification/chest_xray/test')
data_dir_val = pathlib.Path('../input/beginner-chest-xray-image-classification/chest_xray/pred')

In [ ]:
# Checking the number images in all the paths defined above
print("Number of Images in Train:", len(list(data_dir_train.glob("*/*.jpeg"))))
print("Number of Images in Test:", len(list(data_dir_test.glob("*/*.jpeg"))))
print("Number of Images in Validation:", len(list(data_dir_val.glob("*/*.jpeg"))))

In [ ]:
# Here, we are defining some variables which will be commonly used in the further codes
height = 150
width = 150
batch_size = 16
seed = 42 # For reproducibility

<p style="font-size:120%">Inside the folders mentioned above, <strong>we have 2 more folders- "NORMAL" and "PNEUMONIA" respectively!</strong></p>

![image.png](attachment:aeb2a0cd-4164-4868-9be5-999254114924.png)

<p style="font-size:120%">And the images are finally present inside these folders.</p>

![image.png](attachment:56bca59b-060c-45b9-a507-233a8b6e5e49.png)

In [ ]:
# Next! In this step, we are going to use the method "image_dataset_from_directory()" to construct a dataset!

# First we ask Keras to create a "training" dataset with 0.2 as validation split.

train_ds = keras.preprocessing.image_dataset_from_directory(
data_dir_train,
seed=seed,
validation_split=0.2,
subset='training',
image_size=(height,width),
batch_size=batch_size)

In [ ]:
# Second we ask Keras to create a "validation" dataset with 0.2 as validation split.

val_ds = keras.preprocessing.image_dataset_from_directory(
data_dir_train,
seed=seed,
validation_split=0.2,
subset='validation',
image_size=(height,width),
batch_size=batch_size)

In [ ]:
# Finally we are storing all the available class names from the dataset. (NORMAL and PNEUMONIA)

# This also ensures, we have created the dataset successfully!

class_names = train_ds.class_names
class_names

<h1 style="font-size:50px;color:#6166B3"><strong>Visualizing </strong><strong style="color:black">Our Data:</strong></h1>

In [ ]:
# Defining the Canvas size
plt.figure(figsize=[10,8])

# Next we are just picking one image from the unique categories and displaying them:
for index, classes in enumerate(class_names):
    for images in train_ds.file_paths:
        if classes in images:
            img = image.imread(images)
            plt.subplot(1,2,index+1)
            plt.imshow(img, cmap=plt.cm.gist_gray)
            plt.xticks([])
            plt.yticks([])
            plt.title(str(classes))
            break
plt.show()

<h1 style="font-size:50px;color:#6166B3"><strong>Building </strong><strong style="color:black">Our Model:</strong></h1>

<p style="font-size:150%">The below blog is by <strong>Simran Bansari</strong>. Please support and appreciate her <a href="https://medium.datadriveninvestor.com/introduction-to-how-cnns-work-77e0e4cde99b">blog</a>.</p>

<p style="font-size:150%"><strong>Why CNN?</strong></p>

<p style="font-size:120%">One of the main parts of Neural Networks is Convolutional neural networks (CNN). <mark>CNNs use image recognition and classification in order to detect objects, recognize faces, etc. They are made up of neurons with learnable weights and biases.</mark> Each specific neuron receives numerous inputs and then takes a weighted sum over them, where it passes it through an activation function and responds back with an output.</p>

<p style="font-size:120%">CNNs are primarily used to classify images, cluster them by similarities, and then perform object recognition. Many algorithms using CNNs can identify faces, street signs, animals, etc.</p>

<p style="font-size:150%"><strong>How do they work?</strong></p>

<p style="font-size:120%">They are prompt by volume and utilize multi-channeled images. As opposed to flat images that humans can see that only have width and height, CNNs cannot recognize that. Due to digital color images having red-blue-green (RGB) encoding, CNNs mix those three colors to produce the color spectrum humans perceive.</p>

<p style="font-size:120%">A convolutional network ingests such images as three separate strata of color stacked one on top of the other. A normal color image is seen as a rectangular box whose width and height are measured by the number of pixels from those dimensions. The depth layers in the three layers of colours(RGB) interpreted by CNNs are referred to as channels.</p>

![CNN](https://miro.medium.com/max/1400/1*qtinjiZct2w7Dr4XoFixnA.gif)

<p style="font-size:120%"><strong>The first layer</strong> in a CNN network is the <i><strong>CONVOLUTIONAL LAYER</strong></i>, which is the core building block and does most of the computational heavy lifting. Data or imaged is convolved using filters or kernels. Filters are small units that we apply across the data through a sliding window. The depth of the image is the same as the input, for a color image that RGB value of depth is 4, a filter of depth 4 would also be applied to it. This process involves taking the element-wise product of filters in the image and then summing those specific values for every sliding action. The output of a convolution that has a 3d filter with color would be a 2d matrix.</p>

<p style="font-size:120%">Now, the best way to explain a convolutional layer is to imagine a flashlight that is shining over the top left of the image. In order to understand how this works, imagine as if a flashlight shines its light and covers a 5 x 5 area. And now, let’s imagine this flashlight sliding across all the areas of the input image. This flashlight is called a <strong>filter</strong>(or sometimes referred to as a <strong>neuron</strong> or a <strong>kernel</strong>) and the region that it is shining over is called the receptive field. This filter is also an array of numbers (the numbers are called <strong>weights</strong> or <strong>parameters</strong>).</p>

![Convolved](https://miro.medium.com/max/1052/1*GcI7G-JLAQiEoCON7xFbhg.gif)

<p style="font-size:120%"><strong>Second</strong> is the <i><strong>ACTIVATION LAYER</strong></i> which applies the ReLu (Rectified Linear Unit), in this step we apply the rectifier function to increase non-linearity in the CNN. Images are made of different objects that are not linear to each other.</p>

<p style="font-size:120%"><strong>Third</strong>, is the <i><strong>POOLING LAYER</strong></i>, which involves downsampling of features. It is applied through every layer in the 3d volume. Typically there are hyperparameters within this layer:</p>

<ol>
    <li style="font-size:120%"><strong>The dimension of spatial extent:</strong> which is the value of n which we can take N cross and feature representation and map to a single value</li>
    <li style="font-size:120%"><strong>Stride:</strong> which is how many features the sliding window skips along the width and height</li>
</ol>

![Img1](https://miro.medium.com/max/1400/1*lbUtgiANqLoO1GFSc9pHTg.gif)

![Img](https://miro.medium.com/max/1312/1*uUYOqJ-pVFtfOYSyDpxIsw.png)

<p style="font-size:120%">A common <strong>POOLING LAYER</strong> uses a 2 cross 2 max filter with a stride of 2, this is a non-overlapping filter. A max filter would return the max value in the features within the region. Example of max pooling would be when there is 26 across 26 across 32 volume, now by using a max pool layer that has 2 cross 2 filters and astride of 2, the volume would then be reduced to 13 crosses, 13 crosses 32 feature map.</p>

![IMg2](https://miro.medium.com/max/1400/0*Ck5EU5xxHLwo3uiw.jpeg)

<p style="font-size:120%">Lastly, is the <i><strong>FULLY CONNECTED LAYER</strong></i>, which involves <strong>Flattening</strong>. This involves transforming the entire pooled feature map matrix into a single column which is then fed to the neural network for processing. With the fully connected layers, we combined these features together to create a model. Finally, we have an activation function such as softmax or sigmoid to classify the output.</p>

In [ ]:
# Here we start building our model in Keras:

model = Sequential([
  layers.experimental.preprocessing.Rescaling(1./255, input_shape=(height, width, 3)),
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(128, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Dropout(0.2),
  layers.Flatten(),
  layers.Dense(256, activation='relu'),
  layers.Dense(2)
])

In [ ]:
# Next we compile this model where we define the optimizer, loss function and the metric which we will use to evaluate.
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
# Here we will use visualkeras to visualize the CNN Model that we created

# First step: Install Visual Keras
!pip install visualkeras

In [ ]:
# Visualizing our model (Hidden Input)
import visualkeras
visualkeras.layered_view(model, scale_xy=2, legend=True)

In [ ]:
# Finally we are going to train our model for 20 epochs
history = model.fit(train_ds, validation_data=val_ds, epochs=20)

In [ ]:
# Next we are going to plot a graph to check the loss and accuracy as the model trained for 20 epochs for both train and validation.
pd.DataFrame(history.history).plot(figsize=[10,5])
plt.yticks(np.linspace(0,1,6))
plt.xticks(np.linspace(0,10,2))
plt.show()

<h1 style="font-size:50px;color:#6166B3"><strong>Evaluating </strong><strong style="color:black">Our Model:</strong></h1>

In [ ]:
# To evaluate our model, we are going to make use of the "test" dataset
test_ds = keras.preprocessing.image_dataset_from_directory(
data_dir_test,
seed=seed,
image_size=(height,width),
batch_size=batch_size)

In [ ]:
model.evaluate(test_ds, batch_size=batch_size)

<h1 style="font-size:50px;color:#6166B3"><strong>Saving </strong><strong style="color:black">Our Model:</strong></h1>

In [ ]:
# The Next step is to save the model. This is our trained model. We saved it so that we do not need to train it again and again.

# I am saving this in the .h5 format.

model.save("./xray_model.h5")

<h1 style="font-size:50px;color:#6166B3"><strong>Prediction </strong><strong style="color:black">Using Our Model:</strong></h1>

In [ ]:
# Loading the .h5 model that we had saved in the previous step:
my_xray_cnnmodel = keras.models.load_model("./xray_model.h5")

# Defining an image path from the "pred" folder:
image_path = '../input/beginner-chest-xray-image-classification/chest_xray/pred/PNEUMONIA/PNEUMONIA_1.jpeg'

# Preprocessing the image to 150x150x3 size and predicting the label:
image = tf.keras.preprocessing.image.load_img(image_path,target_size=(150,150,3))
input_arr = tf.keras.preprocessing.image.img_to_array(image)
input_arr = np.array([input_arr])
predictions = my_xray_cnnmodel.predict(input_arr)

classes = ['NORMAL', 'PNEUMONIA']

actual = ''

for class_name in classes:
    if class_name in image_path:
        actual = class_name

pred = classes[np.argmax(predictions, axis=1)[0]]

# Finally we are displaying the predicted outcome:
plt.figure(figsize=[8,5])
plt.imshow(image, cmap='gray')
plt.title("Actual:"+actual+" /Predicted:"+pred, size=15)
plt.axis('off')
plt.show()

<p style="text-align:center;font-size:200%;color:#6166B3"><strong>Leave an upvote👍 if you like/fork my work. Comments and Suggestions are Much Appreciated!</strong></p>